In [0]:
from pyspark.sql.functions import to_date, count, sum, avg, round

silver_df = spark.read.table("nyc_taxi_lakehouse.silver.taxi_trips_cleansed")

# Group by pickup date to create daily business KPIs
daily_metrics = (
    silver_df
    .withColumn("trip_date", to_date("tpep_pickup_datetime"))
    .groupBy("trip_date")
    .agg(
        count("*").alias("total_trips"),
        sum("passenger_count").alias("total_passengers"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("fare_amount"), 2).alias("avg_fare_amount"),
        round(avg("trip_distance"), 2).alias("avg_trip_distance")
    )
    .orderBy("trip_date")
)

# Overwrite or merge into the Gold layer
(
    daily_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("nyc_taxi_lakehouse.gold.daily_taxi_metrics")
)

print("Gold daily_taxi_metrics table generated successfully.")

Gold daily_taxi_metrics table generated successfully.


In [0]:
from pyspark.sql.functions import col, when

payment_metrics = (
    silver_df
    .withColumn(
        "payment_method",
        when(col("payment_type") == 1, "Credit Card")
        .when(col("payment_type") == 2, "Cash")
        .when(col("payment_type") == 3, "No Charge")
        .when(col("payment_type") == 4, "Dispute")
        .otherwise("Other")
    )
    .groupBy("payment_method")
    .agg(
        count("*").alias("trip_count"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("tip_amount"), 2).alias("avg_tip")
    )
    .orderBy(col("trip_count").desc())
)

(
    payment_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("nyc_taxi_lakehouse.gold.payment_type_summary")
)

print("Gold payment_type_summary table generated successfully.")

Gold payment_type_summary table generated successfully.


In [0]:
%sql
SELECT * FROM nyc_taxi_lakehouse.gold.daily_taxi_metrics ORDER BY trip_date DESC LIMIT 7;

trip_date,total_trips,total_passengers,total_revenue,avg_fare_amount,avg_trip_distance
2024-02-01,3,7,90.47,19.33,4.08
2024-01-31,95367,120568,2561457.25,17.81,2.99
2024-01-30,94653,120072,2519006.36,17.6,2.98
2024-01-29,79000,101206,2233305.48,18.87,3.46
2024-01-28,82229,115518,2218244.95,18.34,3.43
2024-01-27,102270,146638,2566273.16,17.09,2.86
2024-01-26,97796,131595,2662839.8,18.13,3.13


In [0]:
%sql
SELECT * FROM nyc_taxi_lakehouse.gold.payment_type_summary;

payment_method,trip_count,total_revenue,avg_tip
Credit Card,2273608,6.389132221E7,4.16
Cash,418003,9984553.05,0.0
Dispute,22717,568748.96,0.0
No Charge,9887,220354.96,0.01
